# Strategy 3. Template-based query (2)

Template-based. Create a Query Template that has several different Cypher queries embedded as training cases for the LLM (along with the schema) and then use that as the basis for the query a user adds.

Three options are evaluated:
- 3a - only examples of correct queries are provided
- 3b - examples of correct queries and "normal" graph schema is provided
- 3c - examples of correct queries and "enhanced" graph schema is provided

In [3]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [4]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    elif re.search(r"mistral", model):
        return ChatMistralAI(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]

# models:
#   claude-3-5-sonnet-20240620
#   gpt-4o
#   o1-preview-2024-09-12
#   open-mistral-7b

# llm = ChatModel("gpt-4o")


In [18]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)



In [5]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [19]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

In [8]:
questions = [
    "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)",
    "What is the evidence linking TDP-43 to cancer in animal models?",
    "What (or is there) is the clinical evidence linking BRAF to Melanoma?"
]

## Running template-based query

We'll give few examples of correct cypher queries. All queries will be related to the actual query we want LLM to make. 

- 3a will be prompt that only uses queries
- 3b will be also using graph schema


In [26]:
# Graph schema

from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
graph.refresh_schema()

normal_schema = graph.schema

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `targetInModel`: STRING 
  - `targetInModelMgiId`: STRING 
  - `targetFromSourceId`: STRING 
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
  - `modelPhenotypeLabel`: STRING 
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`

In [52]:
open("normal_schema.txt", "w").write(normal_schema)
open("enhanced_schema.txt", "w").write(enhanced_schema)

25997

In [40]:
from langchain_core.prompts import PromptTemplate

query_example_1 = """\
Task: retreive associations between KRAS and cancer, rna expression only, and evidence score >= 0.1

```cypher
MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = 'KRAS' 
  AND (LOWER(disease.name) CONTAINS ('cancer') OR LOWER(disease.name) CONTAINS ('carcin') OR LOWER(disease.name) CONTAINS ('neoplas'))
  AND assoc.score IS NOT NULL
  AND (assoc:`RnaExpression.GeneToDiseaseAssociation`)
  AND assoc.score >= 0.1
RETURN 
    gene.approvedSymbol as Gene,
    disease.name as Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    assoc.score as Score,
    assoc.literature as Literature
ORDER BY assoc.score DESC
```\
"""

query_example_2 = """\
Task: retreive associations between ALK and non-small cell lung cancer, only known drugs

```cypher
MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = 'ALK' 
  AND (LOWER(disease.name) CONTAINS ('non-small cell lung'))
  AND assoc.score IS NOT NULL
  AND (assoc:`KnownDrug.GeneToDiseaseAssociation`)
RETURN 
    gene.approvedSymbol as Gene,
    disease.name as Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    assoc.score as Score,
    assoc.literature as Literature
ORDER BY assoc.score DESC
```\
"""


query_example_3 = """\
Task: how many evidence of each type there are between KRAS and colorectal cancer?

```cypher
MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = 'KRAS' 
  AND (LOWER(disease.name) CONTAINS ('colorectal cancer'))
  AND assoc.score IS NOT NULL
RETURN 
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    COUNT(*) as Count
ORDER BY Count DESC
```\
"""

examples = f"""\
Here are vew examples:
--------------------------------------------
{query_example_1}

{query_example_2}

{query_example_3}
--------------------------------------------
"""



system_prompt_generic = """
You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a Neo4j database with biological data that accepts queries cypher queries. Scientists will provide you questions, and you should output correct cypher statements that will generate results. 

{schema}
{examples}
"""

normal_schema_description = f"""\
This is graph schema:
--------------------------------------------
{normal_schema}
--------------------------------------------    
"""

enhanced_schema_description = f"""\
This is graph schema:
--------------------------------------------
{enhanced_schema}
--------------------------------------------    
"""


system_prompt_no_schema = system_prompt_generic.format(examples = examples, schema = "")
system_prompt_normal_schema = system_prompt_generic.format(examples = examples, schema = normal_schema_description)
system_prompt_enhanced_schema = system_prompt_generic.format(examples = examples, schema = enhanced_schema_description)

template_query = """
{question}
"""

user_template = PromptTemplate.from_template(template_query)


# Option 3a - withouth schema

3 examples but no graph schema

In [22]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]


def run_llm_3a(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_no_schema}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_no_schema),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_3a(llm_model, question))

Prompting LLM: 100%|██████████| 120/120 [14:22<00:00,  7.18s/it]


In [23]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_3a(llm_model, question)
        time.sleep(2)

In [24]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [00:47<00:00,  2.52it/s]


In [25]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("03a-evaluations.xlsx", index=False)
with open("03a-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],2.535390,0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",0.511704,1444.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",0.292627,1439.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.201686,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[(TARDBP, amyotrophic lateral sclerosis, Genet...",0.287599,1439.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,Task: Retrieve associations between BRAF and M...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.218623,376.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To retrieve the clinical evidence linking BRAF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.195994,376.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.189169,376.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[(BRAF, melanoma, KnownDrug.GeneToDiseaseAssoc...",0.246630,376.0,NaN


# Option 3b - with graph schema in the prompt

One correct query and 2 decoy queries + json examples

In [35]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]


def run_llm_3b(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_normal_schema}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_normal_schema),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_3b(llm_model, question))

Prompting LLM:  91%|█████████ | 109/120 [12:47<00:20,  1.86s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM: 100%|██████████| 120/120 [16:34<00:00,  8.29s/it]


In [36]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_3b(llm_model, question)
        time.sleep(2)

In [37]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [00:58<00:00,  2.04it/s]


In [38]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("03b-evaluations.xlsx", index=False)
with open("03b-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To retrieve the evidence between TDP-43 and am...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.241054,0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",To retrieve the evidence between TDP-43 and am...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.228451,0.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'TARDBP', 'Disease': 'amyotrophic la...",1.320475,1444.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",To retrieve the evidence between TDP-43 and am...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.232043,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.232610,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.343579,304.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.292125,376.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,Task: Retrieve clinical evidence linking BRAF ...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.343658,376.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To determine the clinical evidence linking **B...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.312629,376.0,NaN


# Option 3c - with enhanced graph schema in the prompt


In [41]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]


def run_llm_3c(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_enhanced_schema}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_enhanced_schema),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_3c(llm_model, question))


Prompting LLM:   0%|          | 0/120 [00:00<?, ?it/s]

Prompting LLM:  58%|█████▊    | 70/120 [08:44<01:46,  2.14s/it]

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM: 100%|██████████| 120/120 [17:48<00:00,  8.90s/it]


In [43]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_3c(llm_model, question)
        time.sleep(2)

In [45]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph:   0%|          | 0/120 [00:00<?, ?it/s]

Querying graph: 100%|██████████| 120/120 [00:43<00:00,  2.77it/s]


In [46]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("03c-evaluations.xlsx", index=False)
with open("03c-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To retrieve the evidence between TDP-43 and am...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.226036,0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.227113,0.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.233582,0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",To determine the evidence strength between TDP...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'TARDBP', 'Disease': 'amyotrophic la...",0.667542,1444.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",To determine the strength of evidence between ...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,[],0.223254,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To determine the clinical evidence linking **B...,[{'query': 'MATCH (gene:HumanGene)-[:IS_P...,2,MATCH \n (gene:HumanGene)-[:IS_PART_OF]->(a...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.300606,376.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"Yes, there is clinical evidence linking **BRAF...",[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.302574,376.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking **BRAF**...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.289571,376.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,```cypher\nMATCH (gene:HumanGene)-[:IS_PART_OF...,[{'query': 'MATCH (gene:HumanGene)-[:IS_PART_O...,1,MATCH (gene:HumanGene)-[:IS_PART_OF]->(assoc)<...,True,"[{'Gene': 'BRAF', 'Disease': 'melanoma', 'Evid...",0.277590,376.0,NaN
